# 04A 真实 COF ML 综合案例：CO₂ adsorption

> 🟢 **Level A · 必须掌握** | 综合项目 | 完成标准：独立完成 `data → EDA → clean → features → models → CV → final test → interpretation`。

target 是公开 GCMC 数据，不是实验测量。必须保留条件、单位与 provenance。

In [ ]:
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/gokhanonderaksu/COFSpace/main/OnlyCoRECOF%20-%20Feature%20Sets/CoRECOF%20-%20CO2%20-%201%20BAR.csv')
target='CO2-1 bar (mol/kg)'
features=['PLD (Å)','LCD (Å)','Sacc (m2/g-1)','Porosity','%C','%H','%N','%O','%Metalloid','%Halogen','%Ametal']
X=df[features].copy(); X['LCD_PLD_ratio']=X['LCD (Å)']/X['PLD (Å)']; X=X.fillna(X.median()); y=df[target]
display(df.head()); display(df.describe().T)


In [ ]:
from sklearn.model_selection import KFold,cross_validate,train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor,ExtraTreesRegressor,GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error,r2_score
Xdev,Xtest,ydev,ytest=train_test_split(X,y,test_size=0.2,random_state=42)
cv=KFold(5,shuffle=True,random_state=42)
models={'Ridge':make_pipeline(StandardScaler(),Ridge()),'Random Forest':RandomForestRegressor(n_estimators=400,random_state=42,n_jobs=-1),'Extra Trees':ExtraTreesRegressor(n_estimators=400,random_state=42,n_jobs=-1),'Gradient Boosting':GradientBoostingRegressor(random_state=42)}
rows=[]
for name,m in models.items():
    s=cross_validate(m,Xdev,ydev,cv=cv,scoring={'mae':'neg_mean_absolute_error','r2':'r2'}); rows.append([name,(-s['test_mae']).mean(),(-s['test_mae']).std(),s['test_r2'].mean()])
results=pd.DataFrame(rows,columns=['Model','CV_MAE','CV_MAE_std','CV_R2']).sort_values('CV_MAE'); display(results)
best=models[results.iloc[0]['Model']].fit(Xdev,ydev); p=best.predict(Xtest)
print('test MAE =',mean_absolute_error(ytest,p),'test R² =',r2_score(ytest,p))


## 必做 sensitivity tests
1. 加入 `KCO2 (mol/kg/Pa)` 后重复；2. 换用 0.1/5/10 bar CO₂ 数据；3. 比较 feature importance 随压力变化。

### Dataset card
记录 source/DOI、structures、target、T/P/unit、descriptors、split、preprocessing、model、metrics 和版本。
